In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import glob, os, re

# ---- Load monitoring data ----
path = "datamart/gold/data_monitoring/*.parquet"
files = glob.glob(path)

dfs = []
for f in files:
    df = pd.read_parquet(f)
    # safer extraction for 2024_11_01
    date_match = re.search(r"(\d{4})_(\d{2})_(\d{2})", f)
    if date_match:
        date_str = "-".join(date_match.groups())
        df["snapshot_date"] = pd.to_datetime(date_str)
        dfs.append(df)

monitor_df = pd.concat(dfs).sort_values("snapshot_date").reset_index(drop=True)

# ---- Separate feature-level vs prediction drift ----
monitor_df["type"] = monitor_df["feature"].apply(lambda x: "prediction" if x == "model_score" else "feature")
feature_df = monitor_df[monitor_df["type"] == "feature"]
pred_df = monitor_df[monitor_df["type"] == "prediction"]

print(f"✅ Feature rows: {len(feature_df)}, Prediction rows: {len(pred_df)}")

# =====================================================================
# 1️⃣ FEATURE DRIFT SUMMARY (Aggregated across features)
# =====================================================================
def p(q):
    return lambda s: s.quantile(q)

summary = (
    feature_df
    .groupby("snapshot_date")
    .agg(
        psi_med   = ("psi", "median"),
        psi_p10   = ("psi", p(0.10)),
        psi_p90   = ("psi", p(0.90)),
        ks_med    = ("ks_stat", "median"),
        ks_p10    = ("ks_stat", p(0.10)),
        ks_p90    = ("ks_stat", p(0.90)),
        psi_alert = ("psi",  lambda s: (s > 0.10).sum()),
        psi_major = ("psi",  lambda s: (s > 0.25).sum()),
        ks_p_lt05 = ("ks_pvalue", lambda s: (s < 0.05).mean()*100.0),
        n_feats   = ("psi", "size")
    )
    .reset_index()
)

# ---- Plot PSI & KS stat bands ----
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# PSI (median band)
ax = axes[0]
ax.plot(summary["snapshot_date"], summary["psi_med"], marker="o", label="PSI median")
ax.fill_between(summary["snapshot_date"], summary["psi_p10"], summary["psi_p90"], alpha=0.2, label="PSI p10–p90")
ax.axhline(0.10, ls="--", lw=1, label="PSI=0.10 alert")
ax.axhline(0.25, ls="--", lw=1, label="PSI=0.25 major")
ax.set_title("Feature Drift — PSI (median with p10–p90 band)")
ax.set_ylabel("PSI")
ax.grid(True)
ax.legend()

# KS (median band)
ax = axes[1]
ax.plot(summary["snapshot_date"], summary["ks_med"], marker="o", label="KS stat median")
ax.fill_between(summary["snapshot_date"], summary["ks_p10"], summary["ks_p90"], alpha=0.2, label="KS stat p10–p90")
ax.set_title("Feature Drift — KS Statistic (median with p10–p90 band)")
ax.set_ylabel("KS stat")
ax.grid(True)
ax.legend()

axes[-1].set_xlabel("Snapshot Date")
plt.tight_layout()
plt.show()

# ---- Plot KS p-value alert share ----
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(summary["snapshot_date"], summary["ks_p_lt05"], marker="o")
ax.set_title("Feature Drift — % of features with KS p-value < 0.05")
ax.set_ylabel("% of features")
ax.set_xlabel("Snapshot Date")
ax.grid(True)
plt.tight_layout()
plt.show()

# =====================================================================
# 2️⃣ PREDICTION DRIFT (Single-row per snapshot)
# =====================================================================
if not pred_df.empty:
    pred_df = pred_df.sort_values("snapshot_date")

    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    metrics = ["psi", "ks_stat", "ks_pvalue"]
    titles = [
        "Prediction Drift — PSI (Population Stability Index)",
        "Prediction Drift — KS Statistic",
        "Prediction Drift — KS p-value"
    ]

    for ax, metric, title in zip(axes, metrics, titles):
        ax.plot(pred_df["snapshot_date"], pred_df[metric], marker="o", label=metric)
        ax.set_title(title)
        ax.set_ylabel("Value")
        ax.grid(True)
        ax.legend()

    axes[-1].set_xlabel("Snapshot Date")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No prediction drift records found.")

ValueError: No objects to concatenate